<a href="https://colab.research.google.com/github/mugalan/introduction-to-statistical-learning/blob/main/Gaussian_Process_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Overview

At its core, **Gaussian Process Regression (GPR)** is a way of performing regression that doesn’t just give you a single "best-fit" line, but rather a **distribution of possible functions** that could fit your data.

While traditional linear regression assumes your data fits a specific shape (like $y = mx + b$), a Gaussian Process is "non-parametric"—it is flexible enough to let the data dictate the shape of the curve.

---

1. The "Distribution of Functions" Concept

In basic statistics, a Gaussian (Normal) distribution describes a **random variable** (like the heights of people). A Gaussian Process, however, describes a **random function**.

* Instead of picking a point from a bell curve, you are picking an entire **wiggly line** from a collection of infinite possible lines.
* Before you see any data (the **Prior**), these lines can go anywhere.
* Once you provide data points, the GP "squashes" all those lines so they are forced to pass through (or near) your observations.

2. The Power of Uncertainty

This is the "killer feature" of GPR. Most models give you a prediction but don't tell you how sure they are. GPR provides:

* **The Mean:** The most likely value (the center of the curve).
* **The Standard Deviation:** A "cloud" of uncertainty around the prediction.

> **Key Rule:** The further you move away from your known data points, the wider the uncertainty cloud becomes. The model essentially says, *"I'm guessing the value is X, but I haven't seen any data over here, so I might be way off."*

---

3. The Secret Sauce: Kernels

The behavior of the GP is determined by a **Kernel** (or covariance function). This is a mathematical rule that defines how "related" two points are.

* If two points are close together, the kernel insists their values should be similar (creating a smooth curve).
* If you want a jagged, rough curve instead of a smooth one, you simply change the kernel.

---

4. How it Works (The Workflow)

* **Define a Prior:** You start with an assumption about the data (e.g., "I think the relationship is smooth").
* **Incorporate Data:** You show the model your $(x, y)$ coordinates.
* **Calculate the Posterior:** The model uses Bayesian inference to discard all the functions that don't fit the data.
* **Predict:** You ask the model about a new point. It looks at the nearby data, applies the kernel's logic, and gives you a mean and a confidence interval.

---

Why use it?

* **Small Data Sets:** It works remarkably well when you don't have millions of data points.
* **Optimization:** It's the engine behind **Bayesian Optimization**, used to tune hyperparameters in complex machine learning models.
* **Honesty:** It tells you when it’s guessing, which is vital in high-stakes fields like medicine or engineering.

What kind of data are you looking to model? Knowing the context might help me suggest which kernel would fit your use case best.





# Gaussian Process Regression Problem

Let ${G}$ be some set. Consider a one dimensional Gaussian process $X_g\sim \mathscr{N}(\mu_g,\kappa(g,g))$ where $g\in G$ and $\kappa:G\times G \mapsto \mathbb{R}$ is the covariance (also known as the kernel)
$$\mathbb{E}\left((X_g-\mu_g)(X_{g'}-\mu_{g'})\right)=\kappa(g,g').$$

Let $Y_g$ be another stochastic process that satisfies
\begin{align}
Y_g&=X_g+\nu_g,
\end{align}
where $\nu_g\sim \mathscr{N}(\mu,\sigma_m)$.
This represents a noisy sampling of $X_g$. If one samples the stochastic process $Y_g$ at different $g$ and have the set of observations ${y}_{n}\triangleq [y_{g_1},y_{g_2},\cdots,y_{g_n}]^T$, the Gaussian Process (GP) regression is the problem of estimating $X_g$ given the observations $y_n$.


Consider the multidimensional random variable
$\mathscr{Y}_n\triangleq[Y_{g_1},Y_{g_2},\cdots,Y_{g_n}]^T$ and $\mathscr{X}_n\triangleq[X_{g_1},X_{g_2},\cdots,X_{g_n}]^T$. Then $\mathscr{Y}_n\sim  \mathscr{N}(\mu_n,K_n+\sigma_m^2I)$ where, $\mu_n\triangleq[\mu_{g_1}+\mu,\mu_{g_2}+\mu,\cdots,\mu_{g_n}+\mu]^T$ and $K_n$ is the $n\times n$ covariance matrix
\begin{align*}
K_n&\triangleq \mathbb{E}\left((\mathscr{X}_n-\mu_n)(\mathscr{X}_n-\mu_n)^T\right)=
\begin{bmatrix}
\kappa(g_1,g_1) & \kappa(g_1,g_2) & \cdots & \kappa(g_1,g_n)\\
\vdots & \vdots & \vdots &\vdots\\
\kappa(g_n,g_1) & \kappa(g_n,g_2) & \cdots & \kappa(g_n,g_n)
\end{bmatrix}.
\end{align*}


We then have that
\begin{align*}
\left(\begin{array}{c} X_{g}\\ \mathscr{Y}_n
\end{array}\right)\sim\mathscr{N}\left(\begin{bmatrix}  \mu_{g} \\ \mu_n
\end{bmatrix},\begin{bmatrix} \kappa(g,g) & k^T \\ k & K_n+\sigma^2_m
I \end{bmatrix}\right).
\end{align*}

Thus the Gaussian Process (GP) regression problem boils down to finding $\mathbb{E}[X_{g}|\mathscr{Y}_n]$.


# Solution to the Gaussian Process Regression Problem

We have shown in this note on [Multi-Variate-Gaussians](https://github.com/mugalan/introduction-to-statistical-learning/blob/main/Multivariate_Gaussian_Distributions.ipynb) that the conditional random variable satisfies

$$(X_g \mid \mathscr{Y}_n = y_n) \sim \mathscr{N}\Big(\mu_g + k^T(K_n+\sigma_m^2I)^{-1}(y_n - \mu_n), \quad \kappa(g,g) - k^T(K_n+\sigma_m^2I)^{-1}k\Big).$$


Thus the optimal estimate of $X_g$ given the observations $\mathscr{Y}_n=y_n$ is given by:
$$\mathbb{E}[X_g \mid \mathscr{Y}_n=y_n] = \mu_g + k^T(K_n+\sigma_m^2I)^{-1}(y_n - \mu_n),$$
and the uncertainty is given by
$$\mathrm{Var}(X_g \mid \mathscr{Y}_n=y_n)=\kappa(g,g) - k^T(K_n+\sigma_m^2I)^{-1}k.$$


---

Note that the term $k^T(K_n+\sigma_m^2I)^{-1}$ can be viewed as a matrix of weights $w^T$.
The prediction is then essentially a weighted sum of the observed residuals ($y_n - \mu_n$):
$$\mathbb{E}[X_g \mid y_n] = \mu_g + w^T(y_{n} - \mu_{n}).$$
This shows that GPR is a "Linear Predictor" in the space of the observations.

## Linear regression

###GP - Approach

Consider the stochastic process defined above and let $G=\mathbb{R}^n$,  $\nu\sim \mathscr{N}(0,\sigma_m^2I)$, and $X_g=g^Tw$ for $w\sim\mathscr{N}(\mu_w,\sigma_w^2I)$. Then $\kappa(g,g')=g^T\Sigma_wg'$ and the above expression becomes
\begin{align}
Y_g&=X_g+\nu=g^Tw+\nu.
\end{align}
The GP regression problem reduces in this case to that of the linear regression problem and the above expressions reduce to
\begin{align}
\widehat{\mu}_{g}&=\mu_w+k^T (K_n+\sigma^2_mI)^{-1}(y_n-\mu_n),\\
\sigma^2_{g}&= \sigma^2_m+\kappa(g,g)-k^T (K_n+\sigma^2_mI)^{-1}k.
\end{align}


### Bayesian - Approach

Consider a set of data $\{y_i=y(g_i)\}$ where $G=\mathbb{R}^n$. We will attmpt to model the data by a random process
\begin{align}
Y_g&=g^Tw,
\end{align}
where $X_g=g^Tw$ for $w$ some random variable.
\begin{align}
\mathscr{P}(w\,|\,\{Y_g=y_g\})&\propto\mathscr{P}(\{Y_g=y_g\}\,|\,w)\mathscr{P}(w).
\end{align}

If we assume $w\sim\mathscr{N}(\mu_w,\Sigma_w)$ then from properties of conditional normal distributions
\begin{align*}
p\left(\begin{array}{c}  w \\ \mathscr{Y}_n
\end{array}\right)=\mathscr{N}\left(\begin{bmatrix}  \mu_w \\ \mu_n
\end{bmatrix},\begin{bmatrix} \kappa(g,g) & k^T \\ k & K_n \end{bmatrix}\right).
\end{align*}

\begin{align}
\mathbb{E}\left(\begin{bmatrix} w-\mu_w\\ \mathscr{Y}_n-\mu_n\end{bmatrix}\begin{bmatrix}(w-\mu_w)^T & (\mathscr{Y}_n-\mu_n)^T\end{bmatrix}\right)
\end{align}

Let $\mathscr{P}(w\mid \{\mathscr{Y}_n=y_n\})\sim \mathscr{N}(\widehat{\mu}_{w},\widehat{\Sigma}_{w})$. From the properties of normal distributions (please refer to \cite{} for formulas for constructing the conditional probabilities of Gaussian distributions) we find that

\begin{align}
\widehat{\mu}_{w}&=\mu_w+k^T K_n^{-1}(y_n-\mu_n),\\
\widehat{\Sigma}&= \Sigma_w-k^T K_n^{-1}k.
\end{align}
where
\begin{align}
k&=E\left((\mathscr{Y}_n-\mu_n)(w-\mu_{w})^T\right)=\begin{bmatrix}g_1^T\\\vdots\\ g_n^T\end{bmatrix}E\left((w-\mu_{w})(w-\mu_{w})^T\right)=\begin{bmatrix}g_1^T\\\vdots\\ g_n^T\end{bmatrix}\Sigma_w.
\end{align}
and
\begin{align}
({K_n})_{ij}=E\left((Y_{g_i}-\mu_{g_i})(Y_{g_j}-\mu_{g_j})\right)=g_i^T\Sigma_wg_j
\end{align}